<a href="https://colab.research.google.com/github/Vibsoup/1MS24IS145_assignment2/blob/main/Literatureagentloop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [ ]:
# @title Create a database of classical literature

DOCUMENTS = {
    "the_odyssey": (
        "The Odyssey is an ancient Greek epic poem traditionally attributed to Homer. "
        "It follows Odysseus as he attempts to return home to Ithaca after the Trojan War. "
        "During his journey, he encounters mythical creatures, gods, and dangerous obstacles. "
        "The poem explores themes of loyalty, perseverance, identity, and the meaning of home."
    ),

    "the_iliad": (
        "The Iliad is an ancient Greek epic poem traditionally attributed to Homer. "
        "It focuses on events during the Trojan War, particularly the conflict between Achilles and Agamemnon. "
        "The poem describes battles, relationships between warriors, and the influence of the gods. "
        "Major themes include honor, pride, fate, friendship, and the consequences of war."
    ),

    "divine_comedy": (
        "The Divine Comedy is an Italian epic poem written by Dante Alighieri. "
        "The poem follows Dante's journey through Hell, Purgatory, and Paradise. "
        "Virgil guides Dante through Hell and Purgatory, while Beatrice guides him through Paradise. "
        "The work explores morality, spirituality, justice, love, and the nature of the afterlife."
    ),

    "don_quixote": (
        "Don Quixote is a novel written by Spanish author Miguel de Cervantes. "
        "It follows a man who becomes obsessed with chivalric romances and decides to become a knight. "
        "He travels with his companion Sancho Panza and often mistakes ordinary situations for adventures. "
        "The novel explores imagination, reality, idealism, friendship, and the relationship between stories and life."
    ),

    "pride_and_prejudice": (
        "Pride and Prejudice is a novel written by Jane Austen and published in 1813. "
        "The story follows Elizabeth Bennet as she navigates relationships, family expectations, and social conventions. "
        "Her relationship with Mr. Darcy develops as both characters reconsider their first impressions of each other. "
        "The novel explores love, marriage, class, reputation, prejudice, and personal growth."
    )
}

In [ ]:
# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(
            1 for word in words if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Quick test
print(search_documents("war journey"))


[Source: the_odyssey]
The Odyssey is an ancient Greek epic poem traditionally attributed to Homer. It follows Odysseus as he attempts to return home to Ithaca after the Trojan War. During his journey, he encounters mythical creatures, gods, and dangerous obstacles. The poem explores themes of loyalty, perseverance, identity, and the meaning of home.

[Source: the_iliad]
The Iliad is an ancient Greek epic poem traditionally attributed to Homer. It focuses on events during the Trojan War, particularly the conflict between Achilles and Agamemnon. The poem describes battles, relationships between warriors, and the influence of the gods. Major themes include honor, pride, fate, friendship, and the consequences of war.


In [ ]:
# @title Ask Gemini to search for something

import google.generativeai as genai
import os
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        """
        You are a literature expert.
        When you need specific information about classical literature,
        characters, events, themes, or stories, use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [ ]:
# @title Run the document search agent

import time
from google import genai
from google.genai import types

client = genai.Client()

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the documents to find the most relevant paragraph.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        contents.append(model_content)

        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        result_parts = []

        for call in calls:
            args = dict(call.args)

            print(
                f"Step {step}: model calls "
                f"{call.name}({args})"
            )

            result = search_documents(**args)

            print(
                f"Result preview: "
                f"{str(result)[:100]}..."
            )

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "(Stopped: hit max_steps)"


# Quick test
_ = run_agent(
    "What is the Iliad about?"
)

Question: What is the Iliad about?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Iliad plot summary Achilles Trojan War'})
Result preview: [Source: the_iliad]
The Iliad is an ancient Greek epic poem traditionally attributed to Homer. It fo...

Final answer:
The ***Iliad*** is an ancient Greek epic poem traditionally attributed to Homer. Set during the final year of the Trojan War, it tells the story of a dramatic and pivotal few weeks involving the Greek (Achaean) and Trojan armies. 

Here are the key elements of what the poem is about:

* **The Wrath of Achilles:** The central conflict of the poem is not the entire Trojan War, but rather the rage (**menis**) of Achilles, the greatest Greek warrior. Early in the epic, the Greek commander Agamemnon insults and dishonors Achilles, causing him to withdraw his troops and refuse to fight. 
* **The Turn of the Tide:** Without Achilles on the battlefield, the Greeks begin to suffer heavy de

In [ ]:
# @title Ask follow up questions

questions = [
    "Who is the main character in The Odyssey?",
    "What are the major themes of The Iliad?",
    "Who guides Dante through Hell and Purgatory in The Divine Comedy?",
    # This question is not answered by our database.
    "How many copies of The Odyssey are sold every year?",
    # This question is part of the general knowledge of the model.
    "Who wrote Hamlet?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # Stay within the free-tier rate limit.

    print("\n" + "=" * 60)
    run_agent(q)



Question: Who is the main character in The Odyssey?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'main character in The Odyssey'})
Result preview: [Source: the_odyssey]
The Odyssey is an ancient Greek epic poem traditionally attributed to Homer. I...

Final answer:
The main character in *The Odyssey* is **Odysseus** (also known by his Latin name, Ulysses). He is the Greek king of Ithaca who spends ten years trying to return home to his wife, Penelope, and his son, Telemachus, after the end of the Trojan War.

Question: What are the major themes of The Iliad?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'The Iliad major themes'})
Result preview: [Source: the_iliad]
The Iliad is an ancient Greek epic poem traditionally attributed to Homer. It fo...

Final answer:
Based on *The Iliad*, the major themes of the epic poem include:

* **Honor and Glory (*Kleos*):** Warriors are driven by

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)